# 🛡️ Drishti-Kavach: BiSeNetV2 Rail & Track Segmentation Training (Kaggle)

This notebook trains **BiSeNetV2** for real-time semantic segmentation on the **RailSem19 (Day + 850nm Active NIR Night)** dataset.

### Target Classes:
* `0: Background`
* `1: Track_Bed` (Drivable Track Gauge Corridor)
* `2: Rail_Lines` (Running Steel Rails)

### Recommended Kaggle Settings:
* **Accelerator:** GPU T4 x 2 or GPU P100
* **Persistence:** Files only
* **Internet:** Enabled


In [ ]:
# Cell 1: Hardware & CUDA Verification
import os, sys, time, glob, shutil, random
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('[+] PyTorch Version:', torch.__version__)
print('[+] Execution Device:', device)
if torch.cuda.is_available():
    print('[+] GPU Name:', torch.cuda.get_device_name(0))
    print('[+] GPU Count:', torch.cuda.device_count())
    print(f'[+] VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')


In [ ]:
# Cell 2: Dataset Discovery & Extraction
input_roots = glob.glob('/kaggle/input/**/dataset_segmentation', recursive=True)
zip_files = glob.glob('/kaggle/input/**/*.zip', recursive=True)

DATASET_DIR = None
if input_roots:
    DATASET_DIR = input_roots[0]
    print(f'[+] Found dataset directly at: {DATASET_DIR}')
elif zip_files:
    print(f'[+] Found zip archive: {zip_files[0]}. Extracting to /kaggle/temp/...')
    import zipfile
    os.makedirs('/kaggle/temp', exist_ok=True)
    with zipfile.ZipFile(zip_files[0], 'r') as zf:
        zf.extractall('/kaggle/temp')
    found = glob.glob('/kaggle/temp/**/dataset_segmentation', recursive=True)
    DATASET_DIR = found[0] if found else '/kaggle/temp/dataset_segmentation'
else:
    DATASET_DIR = 'dataset_segmentation'

train_imgs = glob.glob(f'{DATASET_DIR}/images/train/*.jpg')
val_imgs = glob.glob(f'{DATASET_DIR}/images/val/*.jpg')
print(f'[+] Verified Dataset: {len(train_imgs)} Train Frames | {len(val_imgs)} Val Frames')


In [ ]:
# Cell 3: BiSeNetV2 Architecture
class ConvBNReLU(nn.Module):
    def __init__(self, in_chan, out_chan, ks=3, stride=1, padding=1, dilation=1, groups=1, bias=False):
        super(ConvBNReLU, self).__init__()
        self.conv = nn.Conv2d(in_chan, out_chan, kernel_size=ks, stride=stride, padding=padding, dilation=dilation, groups=groups, bias=bias)
        self.bn = nn.BatchNorm2d(out_chan)
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))

class DetailBranch(nn.Module):
    def __init__(self):
        super(DetailBranch, self).__init__()
        self.s1 = nn.Sequential(ConvBNReLU(3, 64, ks=3, stride=2, padding=1), ConvBNReLU(64, 64, ks=3, stride=1, padding=1))
        self.s2 = nn.Sequential(ConvBNReLU(64, 64, ks=3, stride=2, padding=1), ConvBNReLU(64, 64, ks=3, stride=1, padding=1), ConvBNReLU(64, 64, ks=3, stride=1, padding=1))
        self.s3 = nn.Sequential(ConvBNReLU(64, 128, ks=3, stride=2, padding=1), ConvBNReLU(128, 128, ks=3, stride=1, padding=1), ConvBNReLU(128, 128, ks=3, stride=1, padding=1))
    def forward(self, x):
        return self.s3(self.s2(self.s1(x)))

class StemBlock(nn.Module):
    def __init__(self):
        super(StemBlock, self).__init__()
        self.conv_in = ConvBNReLU(3, 16, ks=3, stride=2, padding=1)
        self.left = nn.Sequential(ConvBNReLU(16, 8, ks=1, stride=1, padding=0), ConvBNReLU(8, 16, ks=3, stride=2, padding=1))
        self.right = nn.MaxPool2d(kernel_size=3, stride=2, padding=1, ceil_mode=False)
        self.fuse = ConvBNReLU(32, 16, ks=3, stride=1, padding=1)
    def forward(self, x):
        feat = self.conv_in(x)
        return self.fuse(torch.cat([self.left(feat), self.right(feat)], dim=1))

class GELayer(nn.Module):
    def __init__(self, in_chan, out_chan, stride=1, exp_ratio=6):
        super(GELayer, self).__init__()
        self.stride = stride
        mid_chan = in_chan * exp_ratio
        if stride == 1:
            self.conv = nn.Sequential(
                ConvBNReLU(in_chan, in_chan, ks=3, stride=1, padding=1),
                nn.Conv2d(in_chan, mid_chan, kernel_size=3, stride=1, padding=1, groups=in_chan, bias=False),
                nn.BatchNorm2d(mid_chan), nn.ReLU(inplace=True),
                nn.Conv2d(mid_chan, out_chan, kernel_size=1, stride=1, padding=0, bias=False),
                nn.BatchNorm2d(out_chan)
            )
        else:
            self.conv = nn.Sequential(
                ConvBNReLU(in_chan, in_chan, ks=3, stride=1, padding=1),
                nn.Conv2d(in_chan, mid_chan, kernel_size=3, stride=stride, padding=1, groups=in_chan, bias=False),
                nn.BatchNorm2d(mid_chan),
                nn.Conv2d(mid_chan, mid_chan, kernel_size=3, stride=1, padding=1, groups=mid_chan, bias=False),
                nn.BatchNorm2d(mid_chan), nn.ReLU(inplace=True),
                nn.Conv2d(mid_chan, out_chan, kernel_size=1, stride=1, padding=0, bias=False),
                nn.BatchNorm2d(out_chan)
            )
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_chan, in_chan, kernel_size=3, stride=stride, padding=1, groups=in_chan, bias=False),
                nn.BatchNorm2d(in_chan),
                nn.Conv2d(in_chan, out_chan, kernel_size=1, stride=1, padding=0, bias=False),
                nn.BatchNorm2d(out_chan)
            )
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        if self.stride == 1: return self.relu(self.conv(x) + x)
        else: return self.relu(self.conv(x) + self.shortcut(x))

class CEBlock(nn.Module):
    def __init__(self, in_chan=128, out_chan=128):
        super(CEBlock, self).__init__()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.bn = nn.BatchNorm2d(in_chan)
        self.conv_gap = ConvBNReLU(in_chan, in_chan, ks=1, stride=1, padding=0)
        self.conv_last = ConvBNReLU(in_chan, out_chan, ks=3, stride=1, padding=1)
    def forward(self, x):
        feat = self.conv_gap(self.bn(self.gap(x)))
        return self.conv_last(x + feat)

class SemanticBranch(nn.Module):
    def __init__(self):
        super(SemanticBranch, self).__init__()
        self.s12 = StemBlock()
        self.s3 = nn.Sequential(GELayer(16, 32, stride=2), GELayer(32, 32, stride=1))
        self.s4 = nn.Sequential(GELayer(32, 64, stride=2), GELayer(64, 64, stride=1))
        self.s5 = nn.Sequential(GELayer(64, 128, stride=2), GELayer(128, 128, stride=1), GELayer(128, 128, stride=1), GELayer(128, 128, stride=1), CEBlock(128, 128))
    def forward(self, x):
        f2 = self.s12(x); f3 = self.s3(f2); f4 = self.s4(f3); f5 = self.s5(f4)
        return f2, f3, f4, f5

class BGALayer(nn.Module):
    def __init__(self, detail_chan=128, sem_chan=128, out_chan=128):
        super(BGALayer, self).__init__()
        self.detail_dw = nn.Sequential(nn.Conv2d(detail_chan, detail_chan, 3, 1, 1, groups=detail_chan, bias=False), nn.BatchNorm2d(detail_chan), nn.Conv2d(detail_chan, detail_chan, 1, 1, 0, bias=False))
        self.detail_down = nn.Sequential(ConvBNReLU(detail_chan, detail_chan, ks=3, stride=2, padding=1), nn.AvgPool2d(3, 2, 1))
        self.sem_dw = nn.Sequential(ConvBNReLU(sem_chan, sem_chan, ks=3, stride=1, padding=1), nn.Conv2d(sem_chan, sem_chan, 1, 1, 0, bias=False), nn.Sigmoid())
        self.sem_up = ConvBNReLU(sem_chan, sem_chan, ks=3, stride=1, padding=1)
        self.conv_out = ConvBNReLU(detail_chan, out_chan, ks=3, stride=1, padding=1)
    def forward(self, feat_d, feat_s):
        d_sz = feat_d.size()[2:]
        p1 = self.detail_dw(feat_d) * self.sem_dw(F.interpolate(feat_s, size=d_sz, mode='bilinear', align_corners=False))
        d_down = torch.sigmoid(self.detail_down(feat_d))
        p2 = self.sem_up(feat_s)
        p2_interp = F.interpolate(p2, size=d_down.size()[2:], mode='bilinear', align_corners=False) * d_down
        p2_up = F.interpolate(p2_interp, size=d_sz, mode='bilinear', align_corners=False)
        return self.conv_out(p1 + p2_up)

class SegmentHead(nn.Module):
    def __init__(self, in_chan, mid_chan, num_classes, up_factor=8):
        super(SegmentHead, self).__init__()
        self.conv = ConvBNReLU(in_chan, mid_chan, ks=3, stride=1, padding=1)
        self.drop = nn.Dropout(0.1)
        self.conv_out = nn.Conv2d(mid_chan, num_classes, kernel_size=1, stride=1, padding=0)
        self.up_factor = up_factor
    def forward(self, x, target_size=None):
        logits = self.conv_out(self.drop(self.conv(x)))
        if target_size is not None:
            return F.interpolate(logits, size=target_size, mode='bilinear', align_corners=False)
        elif self.up_factor > 1:
            return F.interpolate(logits, scale_factor=self.up_factor, mode='bilinear', align_corners=False)
        return logits

class BiSeNetV2(nn.Module):
    def __init__(self, num_classes=3, is_training=True):
        super(BiSeNetV2, self).__init__()
        self.is_training = is_training
        self.detail = DetailBranch()
        self.segment = SemanticBranch()
        self.bga = BGALayer(128, 128, 128)
        self.head = SegmentHead(128, 1024, num_classes, up_factor=8)
        if is_training:
            self.aux2 = SegmentHead(16, 64, num_classes, up_factor=4)
            self.aux3 = SegmentHead(32, 128, num_classes, up_factor=8)
            self.aux4 = SegmentHead(64, 256, num_classes, up_factor=16)
            self.aux5 = SegmentHead(128, 512, num_classes, up_factor=32)
    def forward(self, x):
        sz = x.size()[2:]
        fd = self.detail(x); f2, f3, f4, f5 = self.segment(x)
        fuse = self.bga(fd, f5)
        out = self.head(fuse, target_size=sz)
        if self.is_training and self.training:
            return out, self.aux2(f2, target_size=sz), self.aux3(f3, target_size=sz), self.aux4(f4, target_size=sz), self.aux5(f5, target_size=sz)
        return out


In [ ]:
# Cell 4: PyTorch Dataset Loader with Real-Time Augmentation
class RailwaySegmentationDataset(Dataset):
    def __init__(self, root_dir, split='train', img_size=(512, 1024), is_train=True):
        self.root_dir = root_dir
        self.split = split
        self.img_h, self.img_w = img_size
        self.is_train = is_train
        
        self.img_paths = sorted(glob.glob(f'{root_dir}/images/{split}/*.jpg'))
        self.mask_paths = [p.replace('/images/', '/masks/').replace('.jpg', '.png') for p in self.img_paths]
        
        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        self.std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        
    def __len__(self):
        return len(self.img_paths)
        
    def __getitem__(self, idx):
        img = cv2.imread(self.img_paths[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_UNCHANGED)
        
        img = cv2.resize(img, (self.img_w, self.img_h), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, (self.img_w, self.img_h), interpolation=cv2.INTER_NEAREST)
        
        if self.is_train:
            if random.random() > 0.5:
                img = cv2.flip(img, 1)
                mask = cv2.flip(mask, 1)
            if random.random() > 0.5:
                alpha = 1.0 + random.uniform(-0.15, 0.15)
                beta = random.uniform(-15, 15)
                img = np.clip(alpha * img + beta, 0, 255).astype(np.uint8)
                
        img = (img / 255.0 - self.mean) / self.std
        img_tensor = torch.from_numpy(img).permute(2, 0, 1).float()
        mask_tensor = torch.from_numpy(mask).long()
        
        return img_tensor, mask_tensor


In [ ]:
# Cell 5: Loss Function (OHEM Cross-Entropy + Booster Supervision)
class OhemCrossEntropy(nn.Module):
    def __init__(self, thresh=0.7, min_kept=100000, ignore_index=255):
        super(OhemCrossEntropy, self).__init__()
        self.thresh = float(thresh)
        self.min_kept = int(min_kept)
        self.ignore_index = ignore_index
        self.criterion = nn.CrossEntropyLoss(ignore_index=ignore_index, reduction='none')

    def forward(self, predict, target):
        b, c, h, w = predict.size()
        target = target.view(-1)
        valid_mask = target.ne(self.ignore_index)
        target = target * valid_mask.long()
        num_valid = valid_mask.sum()

        prob = F.softmax(predict, dim=1)
        prob = (prob.transpose(0, 1)).reshape(c, -1)

        if self.min_kept > num_valid:
            pass
        elif num_valid > 0:
            prob = prob.masked_fill_(~valid_mask, 1.0)
            mask_prob = prob[target, torch.arange(len(target), dtype=torch.long)]
            threshold = self.thresh
            if self.min_kept > 0:
                index = mask_prob.argsort()
                threshold_index = index[min(len(index), self.min_kept) - 1]
                if mask_prob[threshold_index] > self.thresh:
                    threshold = mask_prob[threshold_index]
                kept_mask = mask_prob.le(threshold)
                target = target * kept_mask.long()
                valid_mask = valid_mask * kept_mask

        target = target.masked_fill_(~valid_mask, self.ignore_index)
        target = target.view(b, h, w)
        loss = self.criterion(predict, target)
        return loss[valid_mask.view(b, h, w)].mean()

class BiSeNetLoss(nn.Module):
    def __init__(self):
        super(BiSeNetLoss, self).__init__()
        self.crit = OhemCrossEntropy(thresh=0.7, min_kept=150000)
    def forward(self, preds, target):
        if isinstance(preds, tuple):
            main_out, aux2, aux3, aux4, aux5 = preds
            l_main = self.crit(main_out, target)
            l_aux2 = self.crit(aux2, target)
            l_aux3 = self.crit(aux3, target)
            l_aux4 = self.crit(aux4, target)
            l_aux5 = self.crit(aux5, target)
            return l_main + 0.4 * (l_aux2 + l_aux3 + l_aux4 + l_aux5)
        return self.crit(preds, target)


In [ ]:
# Cell 6: mIoU Evaluator
class Evaluator:
    def __init__(self, num_classes=3):
        self.num_classes = num_classes
        self.confusion_matrix = np.zeros((num_classes, num_classes))
    
    def add_batch(self, gt, pred):
        mask = (gt >= 0) & (gt < self.num_classes)
        label = self.num_classes * gt[mask].astype(int) + pred[mask]
        count = np.bincount(label, minlength=self.num_classes ** 2)
        self.confusion_matrix += count.reshape(self.num_classes, self.num_classes)
        
    def evaluate(self):
        intersection = np.diag(self.confusion_matrix)
        union = np.sum(self.confusion_matrix, axis=1) + np.sum(self.confusion_matrix, axis=0) - intersection
        ious = intersection / np.maximum(union, 1e-7)
        miou = np.nanmean(ious)
        return miou, ious
        
    def reset(self):
        self.confusion_matrix = np.zeros((self.num_classes, self.num_classes))


In [ ]:
# Cell 7: Full Training Loop with AMP
IMG_SIZE = (512, 1024)   # (H, W)
BATCH_SIZE = 16
EPOCHS = 40
LR = 5e-3
WEIGHT_DECAY = 5e-4
NUM_WORKERS = 4

train_dataset = RailwaySegmentationDataset(DATASET_DIR, split='train', img_size=IMG_SIZE, is_train=True)
val_dataset = RailwaySegmentationDataset(DATASET_DIR, split='val', img_size=IMG_SIZE, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

model = BiSeNetV2(num_classes=3, is_training=True).to(device)
criterion = BiSeNetLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * len(train_loader), eta_min=1e-5)
scaler = GradScaler()
evaluator = Evaluator(num_classes=3)

best_miou = 0.0
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)

print('=' * 75)
print(f' 🛡️ STARTING DRISHTI-KAVACH BISENETV2 TRAINING ({EPOCHS} EPOCHS)')
print(f' • Train Samples: {len(train_dataset)} | Val Samples: {len(val_dataset)}')
print(f' • Resolution:    {IMG_SIZE[1]}x{IMG_SIZE[0]} | Batch Size: {BATCH_SIZE}')
print(f' • Optimizer:     SGD (lr={LR}) + CosineAnnealingLR + AMP FP16')
print('=' * 75)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch [{epoch:02d}/{EPOCHS}]')
    for imgs, masks in pbar:
        imgs = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        
        optimizer.zero_grad()
        with autocast():
            preds = model(imgs)
            loss = criterion(preds, masks)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'lr': f'{scheduler.get_last_lr()[0]:.6f}'})
        
    avg_train_loss = total_loss / len(train_loader)
    
    # Validation Phase
    model.eval()
    evaluator.reset()
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs = imgs.to(device, non_blocking=True)
            with autocast():
                logits = model(imgs)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            evaluator.add_batch(masks.numpy(), preds)
            
    miou, class_ious = evaluator.evaluate()
    
    print(f'\n📊 Epoch [{epoch:02d}/{EPOCHS}] Results:')
    print(f' • Train Loss:       {avg_train_loss:.4f}')
    print(f' • Validation mIoU:  {miou*100:.2f}%')
    print(f'   - Background IoU: {class_ious[0]*100:.2f}%')
    print(f'   - Track Bed IoU:  {class_ious[1]*100:.2f}%')
    print(f'   - Rail Lines IoU: {class_ious[2]*100:.2f}%')
    
    if miou > best_miou:
        best_miou = miou
        torch.save(model.state_dict(), '/kaggle/working/best_bisenetv2_raildrishti.pth')
        print(f' ⭐ NEW BEST MODEL SAVED! (mIoU: {best_miou*100:.2f}%)')
        
    torch.save(model.state_dict(), '/kaggle/working/last_bisenetv2_raildrishti.pth')
    print('-' * 75)


In [ ]:
# Cell 8: Model Export to ONNX
print('[*] Exporting Best Model to ONNX format...')
export_model = BiSeNetV2(num_classes=3, is_training=False).to(device)
pth_path = '/kaggle/working/best_bisenetv2_raildrishti.pth'
if os.path.exists(pth_path):
    export_model.load_state_dict(torch.load(pth_path))
    print('[+] Loaded weights from:', pth_path)
export_model.eval()

dummy_input = torch.randn(1, 3, 512, 1024, device=device)
torch.onnx.export(
    export_model, dummy_input, '/kaggle/working/bisenetv2_raildrishti.onnx',
    input_names=['images'], output_names=['output'],
    dynamic_axes={'images': {0: 'batch'}, 'output': {0: 'batch'}},
    opset_version=14
)
print('[+] Successfully exported: /kaggle/working/bisenetv2_raildrishti.onnx')
